# Rerun 2 (GPU) — tuned fine-tuning comparators for the controlled learning curve

This notebook answers R2-1 and the fine-tuning side of R1-1/R1-2 (Paper ID 20266676). It replaces the single fine-tuning configuration of the first submission (first token, unweighted cross-entropy, no warm-up, at most 5 epochs — which in practice almost always stopped at the epoch cap) with:

1. **A budgeted validation search (pilot)** — 8 full fine-tuning configurations (pooling: first token / masked mean × loss: cross-entropy / class-weighted cross-entropy × learning rate 2e-5 / 5e-5) and 2 LoRA configurations (rank 16 on query/value, mean pooling, class-weighted loss, learning rate 5e-4 / 1e-3). All use linear warm-up (10%) and decay, mixed precision, gradient clipping, at least 300 optimisation steps (10–40 epochs) and early stopping with patience 3. The pilot uses seeds 42 and 7 at 250 and 1,000 training texts and selects **on the validation partition only**; the best full-FT and the best LoRA configuration are then fixed for all runs.
2. **The learning curve in two budget regimes**, matching notebook 03:
   * **A — label-matched** (sizes 100–1,000): the fine-tuned model sees only the *n* training labels — internal stratified 80/20 split for early stopping, then refit on all *n* texts for the selected number of epochs.
   * **B — selection-matched** (sizes 100–3,510): trains on the *n* texts and uses the fixed 809-text validation partition only for early stopping; run for the tuned full fine-tuning and for LoRA.
3. **Full audit trail**: per-epoch histories, selected epoch, all test probabilities, the internal splits of regime A, LoRA adapters of every run, and full checkpoints (fp16) for seed 42 at 100 / 1,000 / 3,510 texts.

The learning-curve subsets are nested and identical to notebook 03 (same `lc_order` function; checked against `lc_subset_order.csv` when the output of notebook 03 is attached).

**Before running on Kaggle**
1. *Add Input*: PRDECT-ID. Optional: the output of notebook 03 (subset cross-check); to resume, the output of the previous version of this notebook.
2. *Settings*: Accelerator **GPU T4 ×1** (or P100), Internet **On**.
3. *Save Version → Save & Run All (Commit)*. A session stops cleanly after `TIME_BUDGET_H` hours (Kaggle limit is 12 h); add the output of that version as input and run again to continue. Expected total GPU time ≈ 12–16 h (printed after the pilot and after every seed).

In [1]:
# ============================================================
# 0. IMPORTS
# ============================================================
import os, sys, re, time, json, glob, random, warnings, gc, platform, shutil, math, copy, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm
import sklearn, transformers
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
SESSION_T0 = time.time()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

torch 2.11.0+cu128 | transformers 5.16.1 | device cuda Tesla T4


## 1. Configuration

In [2]:
# ============================================================
# 1. CONFIG
# ============================================================
SMOKE_TEST = bool(int(os.environ.get("MEXBAL_SMOKE", "0")))

SEEDS   = [42, 7, 13, 77, 101, 202, 303, 404, 505, 606]
SIZES_B = [100, 250, 500, 750, 1000, 1500, 2500, 3510]     # regime B (selection-matched)
SIZES_A = [100, 250, 500, 750, 1000]                        # regime A (label-matched): crossover region
PILOT_SEEDS, PILOT_SIZES = [42, 7], [250, 1000]
VAL_FRAC, TEST_FRAC = 0.15, 0.20

MODEL_NAME = os.environ.get("MEXBAL_FT_MODEL", "intfloat/multilingual-e5-base")
PREFIX, MAX_LEN = "query: ", 128

# ---- training recipe (shared by every configuration) ----------
BATCH, WEIGHT_DECAY, WARMUP_RATIO, CLIP = 16, 0.01, 0.10, 1.0
MIN_STEPS, MIN_EPOCHS, MAX_EPOCHS_CAP, PATIENCE = 300, 10, 40, 3
HEAD_DROPOUT, LORA_DROPOUT, LORA_TARGETS = 0.1, 0.1, ("query", "value")
INNER_VAL_FRAC = 0.20                                      # regime A internal early-stopping split

# ---- budgeted search grid (pilot) -----------------------------
GRID_FULL = [dict(family="full", pooling=p, loss=l, lr=lr)
             for p in ["first", "mean"] for l in ["ce", "wce"] for lr in [2e-5, 5e-5]]
GRID_LORA = [dict(family="lora", pooling="mean", loss="wce", lr=lr, lora_r=16, lora_alpha=32) for lr in [5e-4, 1e-3]]
CONFIG_OVERRIDE = None        # e.g. {"full": {...}, "lora": {...}} to skip the pilot selection

TIME_BUDGET_H = 11.3          # stop cleanly before Kaggle's 12-hour limit
SAVE_FULL_CKPT = {(42, 100), (42, 1000), (42, 3510)}   # (seed, n) of regime-B full-FT runs whose fp16 weights are kept

if SMOKE_TEST:
    SEEDS, SIZES_B, SIZES_A = [42, 7], [100, 300], [100]
    PILOT_SEEDS, PILOT_SIZES = [42], [100]
    GRID_FULL, GRID_LORA = GRID_FULL[:2], GRID_LORA[:1]
    MIN_STEPS, MIN_EPOCHS, MAX_EPOCHS_CAP, PATIENCE = 6, 2, 3, 1
    SAVE_FULL_CKPT = {(42, 100)}

def cfg_id(c):
    s = f"{c['family']}_{c['pooling']}_{c['loss']}_lr{c['lr']:g}"
    return s + (f"_r{c['lora_r']}" if c.get("lora_r") else "")

# ---- paths ------------------------------------------------------
INPUT_ROOTS = ("/kaggle/input", os.environ.get("MEXBAL_DATA", "/nonexistent"))
def _find(pattern, roots=INPUT_ROOTS):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, "**", pattern), recursive=True))
        if hits: return hits[0]
    return None

TOKOPEDIA_CSV = _find("PRDECT-ID Dataset.csv") or _find("*PRDECT*.csv") or _find("Tokopedia Review.csv")
assert TOKOPEDIA_CSV, "PRDECT-ID (Tokopedia) CSV not found: add the dataset as input."
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("MEXBAL_WORK", "./work")
OUT_ROOT = os.path.join(WORK, "rerun_finetune")
_prev = [p for r in INPUT_ROOTS for p in glob.glob(os.path.join(r, "**", "rerun_finetune"), recursive=True) if os.path.isdir(p)]
if _prev and not os.path.exists(OUT_ROOT):
    shutil.copytree(_prev[0], OUT_ROOT); print("Resumed from", _prev[0])
for d in [OUT_ROOT, f"{OUT_ROOT}/predictions", f"{OUT_ROOT}/checkpoints", f"{OUT_ROOT}/lora_adapters"]:
    Path(d).mkdir(parents=True, exist_ok=True)
RUNS_CSV, HIST_CSV, PILOT_CSV = f"{OUT_ROOT}/ft_runs.csv", f"{OUT_ROOT}/ft_history.csv", f"{OUT_ROOT}/pilot_runs.csv"
print("Tokopedia csv:", TOKOPEDIA_CSV, "| output:", OUT_ROOT, "| model:", MODEL_NAME, "| SMOKE_TEST =", SMOKE_TEST)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def append_csv(path, rows):
    new = pd.DataFrame(rows); old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    df = pd.concat([old, new], ignore_index=True); df.to_csv(path, index=False); return df

def done_keys(path, cols):
    if not os.path.exists(path): return set()
    return set(map(tuple, pd.read_csv(path)[cols].astype(str).values.tolist()))

def hours_used(): return (time.time() - SESSION_T0) / 3600

Tokopedia csv: /kaggle/input/datasets/jocelyndumlao/prdect-id-indonesian-emotion-classification/Product Reviews Dataset for Emotions Classification Tasks - Indonesian (PRDECT-ID) Dataset/PRDECT-ID Dataset.csv | output: /kaggle/working/rerun_finetune | model: intfloat/multilingual-e5-base | SMOKE_TEST = False


## 2. Data, splits, nested subsets (identical to notebooks 01 and 03)

In [3]:
def load_tokopedia():
    df = pd.read_csv(TOKOPEDIA_CSV)[["Customer Review", "Emotion"]].dropna().reset_index(drop=True)
    le = LabelEncoder(); df["label"] = le.fit_transform(df["Emotion"])
    df["uid"] = [f"tok_{i:05d}" for i in range(len(df))]
    return df, list(le.classes_)

def _two_stage(splitter, y, seed):
    # identical to the original notebook: 65 / 15 / 20 split, stratified, random_state = seed in both stages
    hold = VAL_FRAC + TEST_FRAC
    tr, h = next(splitter(n_splits=1, test_size=hold, random_state=seed).split(np.zeros(len(y)), y))
    va, te = next(splitter(n_splits=1, test_size=TEST_FRAC / hold, random_state=seed).split(np.zeros(len(h)), y[h]))
    return np.sort(tr), np.sort(h[va]), np.sort(h[te])

LC_ORDER_SALT = 20261023

def lc_order(seed):
    """Nested, approximately stratified ordering of the training partition of one seed.
    The learning-curve subset of size n is the first n entries, so every smaller subset is
    contained in every larger one (nested design). Each class is shuffled and its members are
    spread uniformly over the ranking, so every prefix keeps the class proportions."""
    tr = TOKO_SPLITS[seed][0]; lab = TOKO_DF.label.values[tr]
    rng = np.random.default_rng(LC_ORDER_SALT + seed)
    key = np.empty(len(tr))
    for c in np.unique(lab):
        pos = np.where(lab == c)[0]; m = len(pos)
        key[pos[rng.permutation(m)]] = (np.arange(m) + rng.random(m)) / m
    return np.asarray(tr)[np.argsort(key, kind="stable")]

TOKO_DF, TOKO_CLASSES = load_tokopedia()
K = len(TOKO_CLASSES); Y = TOKO_DF.label.values
TOKO_SPLITS = {s: _two_stage(StratifiedShuffleSplit, Y, s) for s in SEEDS}
LC_ORDER = {s: lc_order(s) for s in SEEDS}

# cross-check with notebook 03 (if its output is attached)
_ord = _find("lc_subset_order.csv")
if _ord:
    o = pd.read_csv(_ord)
    for s in SEEDS:
        ref = o[o.seed == s].sort_values("rank").uid.values
        if len(ref): assert np.array_equal(ref, TOKO_DF.uid.values[LC_ORDER[s]]), f"subset order differs for seed {s}"
    print("Subset order identical to notebook 03:", _ord)

def inner_split(sub, seed, n):
    # regime A: stratified 80/20 split of the n training texts (early stopping only)
    a, b = next(StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC, random_state=seed * 10000 + n)
                .split(np.zeros(len(sub)), Y[sub]))
    return np.asarray(sub)[a], np.asarray(sub)[b]

rows = []
for s in SEEDS:
    for n in SIZES_A:
        a, b = inner_split(LC_ORDER[s][:n], s, n)
        rows += [{"seed": s, "train_size": n, "part": "inner_train", "uid": u} for u in TOKO_DF.uid.values[a]]
        rows += [{"seed": s, "train_size": n, "part": "inner_val", "uid": u} for u in TOKO_DF.uid.values[b]]
pd.DataFrame(rows).to_csv(f"{OUT_ROOT}/regimeA_inner_splits.csv", index=False)
print("Tokopedia:", len(TOKO_DF), TOKO_CLASSES, "| seed 42 train/val/test:", [len(x) for x in TOKO_SPLITS[42]])

Tokopedia: 5400 ['Anger', 'Fear', 'Happy', 'Love', 'Sadness'] | seed 42 train/val/test: [3510, 809, 1081]


## 3. Tokenisation, model (first-token / mean pooling, optional LoRA), training loop

In [4]:
# ============================================================
# 3. MODEL + TRAINING
# ============================================================
TOK = AutoTokenizer.from_pretrained(MODEL_NAME)
PAD_ID = TOK.pad_token_id
ENC_IDS = TOK([PREFIX + str(t) for t in TOKO_DF["Customer Review"].values], truncation=True, max_length=MAX_LEN)["input_ids"]
BASE_ENC = AutoModel.from_pretrained(MODEL_NAME)            # loaded once, deep-copied for every run
print("mean tokens per review:", round(float(np.mean([len(x) for x in ENC_IDS])), 1))

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale

def inject_lora(module, r, alpha, dropout, targets):
    n = 0
    for _, parent in list(module.named_modules()):
        for child_name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and child_name in targets:
                setattr(parent, child_name, LoRALinear(child, r, alpha, dropout)); n += 1
    return n

def masked_mean(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

class Classifier(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.enc = copy.deepcopy(BASE_ENC); self.pooling = cfg["pooling"]
        if cfg.get("lora_r"):
            for p in self.enc.parameters(): p.requires_grad = False
            n = inject_lora(self.enc, cfg["lora_r"], cfg["lora_alpha"], LORA_DROPOUT, LORA_TARGETS)
            assert n == 2 * self.enc.config.num_hidden_layers, f"LoRA injected into {n} modules"
        self.drop = nn.Dropout(HEAD_DROPOUT); self.head = nn.Linear(self.enc.config.hidden_size, K)
    def forward(self, ids, mask):
        h = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        z = h[:, 0] if self.pooling == "first" else masked_mean(h, mask)
        return self.head(self.drop(z))

def batches(idx, bs, rng=None):
    idx = np.asarray(idx)
    if rng is not None: idx = idx[rng.permutation(len(idx))]
    for i in range(0, len(idx), bs): yield idx[i:i + bs]

def to_tensors(b):
    seqs = [ENC_IDS[i] for i in b]; L = max(map(len, seqs))
    ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long); mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for k, s in enumerate(seqs): ids[k, :len(s)] = torch.tensor(s); mask[k, :len(s)] = 1
    return ids.to(DEVICE), mask.to(DEVICE)

def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP) if USE_AMP else torch.autocast("cpu", enabled=False)

@torch.no_grad()
def predict_proba(model, idx, bs=64):
    model.eval(); idx = np.asarray(idx)
    order = np.argsort([len(ENC_IDS[i]) for i in idx], kind="stable"); out = []
    for b in batches(idx[order], bs):
        ids, mask = to_tensors(b)
        with autocast(): logits = model(ids, mask)
        out.append(torch.softmax(logits.float(), -1).cpu().numpy())
    P = np.vstack(out); res = np.empty_like(P); res[order] = P; return res

macro = lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0)

def trainable_state(model):
    keys = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keys}

def fit(cfg, train_idx, sel_idx, seed, fixed_epochs=None):
    """Train on train_idx. With sel_idx: early stopping on its Macro F1 (best epoch restored).
    With fixed_epochs: train exactly that many epochs (refit in regime A)."""
    set_seed(seed)
    model = Classifier(cfg).to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    weight = None
    if cfg["loss"] == "wce":
        cnt = np.bincount(Y[train_idx], minlength=K)
        weight = torch.tensor(len(train_idx) / (K * np.maximum(cnt, 1)), dtype=torch.float, device=DEVICE)
    steps = math.ceil(len(train_idx) / BATCH)
    E = int(fixed_epochs) if fixed_epochs else int(np.clip(math.ceil(MIN_STEPS / steps), MIN_EPOCHS, MAX_EPOCHS_CAP))
    total, warm = E * steps, max(1, int(WARMUP_RATIO * E * steps))
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=WEIGHT_DECAY)
    sched = LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    rng = np.random.default_rng(seed); hist = []; best_f, best_ep, best_state, bad = -1.0, 0, None, 0
    t0 = time.perf_counter()
    for ep in range(1, E + 1):
        model.train(); tl, nb = 0.0, 0
        for b in batches(train_idx, BATCH, rng):
            ids, mask = to_tensors(b); yt = torch.tensor(Y[b], device=DEVICE)
            with autocast(): logits = model(ids, mask)
            loss = F.cross_entropy(logits.float(), yt, weight=weight)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, CLIP)
            scaler.step(opt); scaler.update(); sched.step(); tl += loss.item(); nb += 1
        rec = {"epoch": ep, "train_loss": tl / nb, "lr_end": sched.get_last_lr()[0], "elapsed_s": time.perf_counter() - t0}
        if sel_idx is not None:
            f = macro(Y[sel_idx], predict_proba(model, sel_idx).argmax(1)); rec["sel_macro_f1"] = f
            if f > best_f + 1e-6: best_f, best_ep, best_state, bad = f, ep, trainable_state(model), 0
            else: bad += 1
        hist.append(rec)
        if sel_idx is not None and bad >= PATIENCE: break
    if sel_idx is not None: model.load_state_dict(best_state, strict=False)
    else: best_ep = E
    info = {"best_sel_macro_f1": best_f if sel_idx is not None else np.nan, "best_epoch": best_ep,
            "epochs_run": len(hist), "max_epochs": E, "steps_per_epoch": steps,
            "stopped_at_cap": bool(len(hist) == E and bad < PATIENCE) if sel_idx is not None else None,
            "train_time_s": time.perf_counter() - t0}
    return model, info, hist

def free(model):
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

mean tokens per review: 29.8


## 4. Pilot: budgeted validation search (seeds 42, 7; 250 and 1,000 texts; validation partition only)

In [5]:
# ============================================================
# 4. PILOT
# ============================================================
done = done_keys(PILOT_CSV, ["config_id", "seed", "train_size"])
for seed in PILOT_SEEDS:
    va = TOKO_SPLITS[seed][1]
    for n in PILOT_SIZES:
        sub = LC_ORDER[seed][:n]
        for cfg in GRID_FULL + GRID_LORA:
            if (cfg_id(cfg), str(seed), str(n)) in done: continue
            model, info, hist = fit(cfg, sub, va, seed); free(model)
            append_csv(PILOT_CSV, [{"config_id": cfg_id(cfg), **cfg, "seed": seed, "train_size": n, **info}])
            append_csv(HIST_CSV, [{"run": f"pilot|{cfg_id(cfg)}|{seed}|{n}", **h} for h in hist])
            print(f"pilot {cfg_id(cfg):32s} seed={seed} n={n}: val F1={info['best_sel_macro_f1']:.4f} "
                  f"best_ep={info['best_epoch']}/{info['epochs_run']} ({info['train_time_s']:.0f}s)")

pilot = pd.read_csv(PILOT_CSV)
summary = (pilot.groupby(["family", "config_id"]).agg(val_macro_f1=("best_sel_macro_f1", "mean"), runs=("seed", "size"),
           mean_time_s=("train_time_s", "mean")).reset_index().sort_values(["family", "val_macro_f1"], ascending=[True, False]))
summary.to_csv(f"{OUT_ROOT}/pilot_summary.csv", index=False); print(summary.round(4).to_string(index=False))
SEL_PATH = f"{OUT_ROOT}/selected_configs.json"
if CONFIG_OVERRIDE: SELECTED = CONFIG_OVERRIDE
elif os.path.exists(SEL_PATH): SELECTED = json.load(open(SEL_PATH))       # never re-select after resuming
else:
    grids = {"full": GRID_FULL, "lora": GRID_LORA}
    SELECTED = {fam: next(c for c in grids[fam] if cfg_id(c) == summary[summary.family == fam].config_id.iloc[0])
                for fam in ["full", "lora"]}
json.dump(SELECTED, open(SEL_PATH, "w"), indent=2); print("Selected:", {k: cfg_id(v) for k, v in SELECTED.items()})

# rough projection of the remaining GPU time: t(n) = a + b*n per family, fitted on the pilot
def _proj(fam):
    g = pilot[pilot.family == fam].groupby("train_size").train_time_s.mean()
    if len(g) < 2: return lambda n: float(g.iloc[0]) * n / g.index[0]
    b = (g.iloc[-1] - g.iloc[0]) / (g.index[-1] - g.index[0]); a = g.iloc[0] - b * g.index[0]
    return lambda n: max(10.0, a + b * n)
T_FULL, T_LORA = _proj("full"), _proj("lora")
per_seed = sum(T_FULL(n) for n in SIZES_B) + sum(T_LORA(n) for n in SIZES_B) + sum(1.8 * T_FULL(n) for n in SIZES_A)
print(f"Projected main-run time: {per_seed / 60:.0f} min per seed, {len(SEEDS) * per_seed / 3600:.1f} h for {len(SEEDS)} seeds "
      f"(+{pilot.train_time_s.sum() / 3600:.1f} h pilot). Used in this session so far: {hours_used():.2f} h")

pilot full_first_ce_lr2e-05            seed=42 n=250: val F1=0.6145 best_ep=10/13 (53s)
pilot full_first_ce_lr5e-05            seed=42 n=250: val F1=0.6181 best_ep=7/10 (38s)
pilot full_first_wce_lr2e-05           seed=42 n=250: val F1=0.6102 best_ep=10/13 (51s)
pilot full_first_wce_lr5e-05           seed=42 n=250: val F1=0.6237 best_ep=9/12 (46s)
pilot full_mean_ce_lr2e-05             seed=42 n=250: val F1=0.6129 best_ep=10/13 (51s)
pilot full_mean_ce_lr5e-05             seed=42 n=250: val F1=0.6175 best_ep=8/11 (41s)
pilot full_mean_wce_lr2e-05            seed=42 n=250: val F1=0.6046 best_ep=10/13 (51s)
pilot full_mean_wce_lr5e-05            seed=42 n=250: val F1=0.6099 best_ep=5/8 (31s)
pilot lora_mean_wce_lr0.0005_r16       seed=42 n=250: val F1=0.6358 best_ep=17/19 (23s)
pilot lora_mean_wce_lr0.001_r16        seed=42 n=250: val F1=0.6337 best_ep=12/15 (18s)
pilot full_first_ce_lr2e-05            seed=42 n=1000: val F1=0.6320 best_ep=3/6 (73s)
pilot full_first_ce_lr5e-05           

## 5. Learning curve: regime A (label-matched) and regime B (selection-matched)

In [6]:
# ============================================================
# 5. MAIN LOOP (seed-major, so every finished seed is complete; resumable; stops before the time budget)
# ============================================================
def record(regime, comparator, cfg, seed, n, model, info, hist, extra=None):
    te = TOKO_SPLITS[seed][2]; P = predict_proba(model, te); yp = P.argmax(1); yt = Y[te]
    row = {"regime": regime, "comparator": comparator, "config_id": cfg_id(cfg), "seed": seed, "train_size": n,
           "macro_f1": macro(yt, yp), "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
           "accuracy": accuracy_score(yt, yp), "n_pred_classes": int(len(np.unique(yp))), **info, **(extra or {})}
    tag = f"{regime}_{comparator.split()[0]}_seed{seed}_n{n}"
    np.savez_compressed(f"{OUT_ROOT}/predictions/{tag}.npz", proba=P.astype(np.float16), test_uid=TOKO_DF.uid.values[te])
    if cfg.get("lora_r"): torch.save(trainable_state(model), f"{OUT_ROOT}/lora_adapters/{tag}.pt")
    elif regime.startswith("B") and (seed, n) in SAVE_FULL_CKPT:
        torch.save({k: v.half() for k, v in model.state_dict().items()}, f"{OUT_ROOT}/checkpoints/{tag}_fp16.pt")
    append_csv(RUNS_CSV, [row]); append_csv(HIST_CSV, [{"run": f"{tag}|{h_tag}", **h} for h_tag, hs in hist for h in hs])
    print(f"{regime:20s} {comparator:13s} seed={seed:<3} n={n:<5} macroF1={row['macro_f1']:.4f} "
          f"best_ep={info['best_epoch']}/{info['max_epochs']} ({info['train_time_s']:.0f}s) | {hours_used():.2f} h used")

stop = False
for seed in SEEDS:
    if stop: break
    tr, va, te = TOKO_SPLITS[seed]; done = done_keys(RUNS_CSV, ["regime", "comparator", "seed", "train_size"])
    jobs = [("A_label_matched", "FT (tuned)", n) for n in SIZES_A] + \
           [("B_selection_matched", "FT (tuned)", n) for n in SIZES_B] + \
           [("B_selection_matched", "LoRA (tuned)", n) for n in SIZES_B]
    for regime, comp, n in jobs:
        if (regime, comp, str(seed), str(n)) in done: continue
        est = (1.8 * T_FULL(n) if regime.startswith("A") else (T_LORA(n) if comp.startswith("LoRA") else T_FULL(n))) / 3600
        if hours_used() + est > TIME_BUDGET_H:
            print(f"Time budget reached ({hours_used():.2f} h). Re-run with this version's output attached to continue."); stop = True; break
        cfg = SELECTED["lora"] if comp.startswith("LoRA") else SELECTED["full"]
        sub = LC_ORDER[seed][:n]
        if regime.startswith("A"):
            itr, iva = inner_split(sub, seed, n)
            m1, info1, h1 = fit(cfg, itr, iva, seed); free(m1)
            model, info, h2 = fit(cfg, sub, None, seed, fixed_epochs=info1["best_epoch"])
            extra = {"inner_best_epoch": info1["best_epoch"], "inner_epochs_run": info1["epochs_run"],
                     "inner_max_epochs": info1["max_epochs"], "inner_stopped_at_cap": info1["stopped_at_cap"],
                     "inner_val_macro_f1": info1["best_sel_macro_f1"], "selection_time_s": info1["train_time_s"]}
            record(regime, comp, cfg, seed, n, model, info, [("inner", h1), ("refit", h2)], extra)
        else:
            model, info, h = fit(cfg, sub, va, seed)
            record(regime, comp, cfg, seed, n, model, info, [("main", h)])
        free(model)
    if not stop: print(f"=== seed {seed} complete | {hours_used():.2f} h used ===")

A_label_matched      FT (tuned)    seed=42  n=100   macroF1=0.1462 best_ep=1/1 (1s) | 0.67 h used
A_label_matched      FT (tuned)    seed=42  n=250   macroF1=0.4843 best_ep=4/4 (11s) | 0.68 h used
A_label_matched      FT (tuned)    seed=42  n=500   macroF1=0.6417 best_ep=7/7 (39s) | 0.70 h used
A_label_matched      FT (tuned)    seed=42  n=750   macroF1=0.6624 best_ep=6/6 (50s) | 0.74 h used
A_label_matched      FT (tuned)    seed=42  n=1000  macroF1=0.6427 best_ep=4/4 (44s) | 0.77 h used
B_selection_matched  FT (tuned)    seed=42  n=100   macroF1=0.5080 best_ep=14/40 (39s) | 0.78 h used
B_selection_matched  FT (tuned)    seed=42  n=250   macroF1=0.5801 best_ep=10/19 (52s) | 0.79 h used
B_selection_matched  FT (tuned)    seed=42  n=500   macroF1=0.6570 best_ep=6/10 (61s) | 0.81 h used
B_selection_matched  FT (tuned)    seed=42  n=750   macroF1=0.6412 best_ep=5/10 (75s) | 0.83 h used
B_selection_matched  FT (tuned)    seed=42  n=1000  macroF1=0.6684 best_ep=7/10 (122s) | 0.87 h used
B_s

## 6. Summary, environment and archive

In [7]:
# ============================================================
# 6. SUMMARY + ENVIRONMENT + ZIP
# ============================================================
if os.path.exists(RUNS_CSV):
    r = pd.read_csv(RUNS_CSV)
    print(r.pivot_table(index=["regime", "comparator"], columns="train_size", values="macro_f1", aggfunc="mean").round(4))
    print(r.groupby(["regime", "comparator"]).agg(runs=("seed", "size"), seeds=("seed", "nunique"),
          stopped_at_cap=("stopped_at_cap", "mean"), mean_best_epoch=("best_epoch", "mean")).round(2))
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "amp_fp16": USE_AMP,
       "model": MODEL_NAME, "prefix": PREFIX, "max_len": MAX_LEN, "batch": BATCH, "weight_decay": WEIGHT_DECAY,
       "warmup_ratio": WARMUP_RATIO, "clip": CLIP, "min_steps": MIN_STEPS, "min_epochs": MIN_EPOCHS,
       "max_epochs_cap": MAX_EPOCHS_CAP, "patience": PATIENCE, "lora_targets": LORA_TARGETS,
       "grid_full": [cfg_id(c) for c in GRID_FULL], "grid_lora": [cfg_id(c) for c in GRID_LORA],
       "selected": {k: cfg_id(v) for k, v in SELECTED.items()}, "seeds": SEEDS, "sizes_A": SIZES_A, "sizes_B": SIZES_B,
       "pilot_seeds": PILOT_SEEDS, "pilot_sizes": PILOT_SIZES, "smoke_test": SMOKE_TEST,
       "session_hours": round(hours_used(), 2)}
json.dump(env, open(f"{OUT_ROOT}/environment_gpu.json", "w"), indent=2)
try:
    open(f"{OUT_ROOT}/pip_freeze_gpu.txt", "w").write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
except Exception as e: print("pip freeze failed:", e)
shutil.make_archive(f"{WORK}/rerun_finetune_outputs", "zip", OUT_ROOT)
print("Archive:", f"{WORK}/rerun_finetune_outputs.zip")

train_size                          100     250     500     750     1000  \
regime              comparator                                             
A_label_matched     FT (tuned)    0.3600  0.5434  0.5942  0.6236  0.6422   
B_selection_matched FT (tuned)    0.4893  0.5577  0.6074  0.6319  0.6437   
                    LoRA (tuned)  0.5056  0.5680  0.6221  0.6352  0.6512   

train_size                          1500    2500    3510  
regime              comparator                            
A_label_matched     FT (tuned)       NaN     NaN     NaN  
B_selection_matched FT (tuned)    0.6531  0.6752  0.6793  
                    LoRA (tuned)  0.6441  0.6714  0.6742  
                                  runs  seeds  stopped_at_cap  mean_best_epoch
regime              comparator                                                
A_label_matched     FT (tuned)      50     10             NaN             7.12
B_selection_matched FT (tuned)      80     10            0.10             6.82
        